# Baseline XGBoost Model (Feature-Only)

This notebook trains an XGBoost classifier using only engineered features from `clean_data/train/` and `clean_data/test/` (no raw sequence one-hot arrays).

- Load full training split from `clean_data/train`
- Create an in-memory 10% validation split from train (not saved)
- Use validation for early stopping each boosting round
- Report final accuracy on `test`

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [ ]:
BASE_DIR = Path('.')
CLEAN_DIR = BASE_DIR / 'clean_data'

FEATURE_PATHS = {
    'train': CLEAN_DIR / 'train' / 'features.csv',
    'test': CLEAN_DIR / 'test' / 'features.csv',
}

TARGET_PATHS = {
    'train': CLEAN_DIR / 'train' / 'targets.csv',
    'test': CLEAN_DIR / 'test' / 'targets.csv',
}

INDEX_COL = 'row_index'
TARGET_COL = 'class_index'

In [5]:
def load_split(split_name: str):
    feat_df = pd.read_csv(FEATURE_PATHS[split_name])
    tgt_df = pd.read_csv(TARGET_PATHS[split_name])

    # These files can contain duplicate row_index values, so align by row order
    # after validating basic consistency.
    if len(feat_df) != len(tgt_df):
        raise ValueError(
            f"Row count mismatch for {split_name}: "
            f"features={len(feat_df)}, targets={len(tgt_df)}"
        )

    if INDEX_COL in feat_df.columns and INDEX_COL in tgt_df.columns:
        if not feat_df[INDEX_COL].equals(tgt_df[INDEX_COL]):
            raise ValueError(
                f"Index order mismatch in {split_name}. "
                f"Regenerate clean_data to restore alignment."
            )

    feature_cols = [c for c in feat_df.columns if c != INDEX_COL]
    X = feat_df[feature_cols].to_numpy(dtype=np.float32)
    y = tgt_df[TARGET_COL].to_numpy(dtype=np.int32)

    return X, y, feature_cols


X_full_train, y_full_train, feature_cols = load_split('train')
X_test, y_test, _ = load_split('test')

# Create an in-memory 10% validation split from training data only.
class_counts = np.bincount(y_full_train)
can_stratify = np.all(class_counts[class_counts > 0] >= 2)
stratify_target = y_full_train if can_stratify else None

X_train, X_val, y_train, y_val = train_test_split(
    X_full_train,
    y_full_train,
    test_size=0.10,
    random_state=42,
    shuffle=True,
    stratify=stratify_target,
)

num_classes = int(np.max(y_full_train) + 1)

print('Full train:', X_full_train.shape, y_full_train.shape)
print('Train (90%):', X_train.shape, y_train.shape)
print('Validation (10%):', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)
print('Stratified split:', can_stratify)
print('Num classes:', num_classes)
print('Feature columns:', feature_cols)

Train: (22593, 13) (22593,)
Validation: (4577, 13) (4577,)
Test: (8326, 13) (8326,)
Num classes: 10
Feature columns: ['length', 'pct_A', 'pct_T', 'pct_C', 'pct_G', 'first_50_pA', 'first_50_pT', 'first_50_pC', 'first_50_pG', 'last_50_pA', 'last_50_pT', 'last_50_pC', 'last_50_pG']


In [6]:
# Extra trees per boosting round with num_parallel_tree > 1
# Early stopping uses validation set every boosting round.
model = XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    num_parallel_tree=2,
    tree_method='hist',
    eval_metric='mlogloss',
    early_stopping_rounds=100,
    random_state=42,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50,
)

print('Best iteration:', model.best_iteration)
print('Best validation score:', model.best_score)

[0]	validation_0-mlogloss:1.39380	validation_1-mlogloss:1.35402
[50]	validation_0-mlogloss:0.65950	validation_1-mlogloss:0.67092
[100]	validation_0-mlogloss:0.48732	validation_1-mlogloss:0.50292
[150]	validation_0-mlogloss:0.41557	validation_1-mlogloss:0.43120
[200]	validation_0-mlogloss:0.37503	validation_1-mlogloss:0.38918
[250]	validation_0-mlogloss:0.34560	validation_1-mlogloss:0.35876
[300]	validation_0-mlogloss:0.32226	validation_1-mlogloss:0.33457
[350]	validation_0-mlogloss:0.30222	validation_1-mlogloss:0.31354
[400]	validation_0-mlogloss:0.28377	validation_1-mlogloss:0.29410
[450]	validation_0-mlogloss:0.26652	validation_1-mlogloss:0.27594
[500]	validation_0-mlogloss:0.25068	validation_1-mlogloss:0.25955
[550]	validation_0-mlogloss:0.23607	validation_1-mlogloss:0.24432
[600]	validation_0-mlogloss:0.22218	validation_1-mlogloss:0.22999
[650]	validation_0-mlogloss:0.20948	validation_1-mlogloss:0.21674
[700]	validation_0-mlogloss:0.19755	validation_1-mlogloss:0.20434
[750]	validat

In [7]:
# Final test-set accuracy block
y_pred_test = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)

print(f'Test accuracy: {test_acc:.4f}')
print('\nClassification report (test):')
print(classification_report(y_test, y_pred_test, digits=4))

Test accuracy: 0.9772

Classification report (test):
              precision    recall  f1-score   support

           0     0.9920    0.9868    0.9894      2651
           1     0.9630    0.9774    0.9701       133
           2     0.9620    0.8261    0.8889       184
           3     0.9701    0.9889    0.9794      3800
           4     0.9767    0.9374    0.9566       894
           5     0.9730    1.0000    0.9863        72
           6     0.0000    0.0000    0.0000         1
           7     0.9143    0.8421    0.8767        38
           8     0.9656    0.9704    0.9680       405
           9     0.9797    0.9797    0.9797       148

    accuracy                         0.9772      8326
   macro avg     0.8696    0.8509    0.8595      8326
weighted avg     0.9771    0.9772    0.9769      8326



/Users/aryamanwade/Desktop/ml_projects/dna_sequencing/dna-sequencing/ml_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aryamanwade/Desktop/ml_projects/dna_sequencing/dna-sequencing/ml_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aryamanwade/Desktop/ml_projects/dna_sequencing/dna-sequencing/ml_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels